# DRC — Smoke test (~10 min): verify the pipeline runs on Kaggle

**Run this first.** It exercises the *entire* pipeline — every stage, data through figures — but on a tiny config (`configs/smoke.yaml`): a ~150k-word slice of the corpus, a small 2-layer model, 1 epoch, 1 seed, and a handful of doses. It finishes in roughly **10 minutes on dual-T4** and exists to catch environment/wiring errors *before* you commit to the ~10-hour real run. It also exercises the fp16 path T4 needs. Its outputs go to **separate folders** (`data/smoke/`, `models/smoke/`, `results/smoke/`), so they never collide with or pollute the real run. If the dashboard and health check come back clean here, the full run should too.

**This notebook is resumable.** Every stage runs in its own subprocess via
`drc.pipeline.run_pipeline`, so a CUDA OOM or a hard kernel crash in one stage
takes down that subprocess and *nothing else* — the kernel survives, the
failure is recorded, and the rest of the pipeline proceeds where it can. If
Kaggle's 12-hour session limit cuts you off, just **re-run the notebook**:
finished stages detect their own outputs and skip, so you pick up where you
stopped instead of redoing hours of work.

**Scope:** every stage, on the tiny `configs/smoke.yaml`.

## Before you run

- Set the accelerator to **GPU T4 x2** (Settings -> Accelerator -> *GPU T4 x2*).
  With two cards the training sweep runs two jobs in parallel; with one it falls
  back to a 48-run single-GPU plan.
- **T4 is a Turing GPU and does not support bf16.** The shipped
  `configs/base.yaml` sets `precision: bf16`. Before training on T4, change that
  line to `precision: fp16`. The GPU-detect cell below reminds you; it does not
  edit the config for you.

## Two notebooks

The pipeline is split into just two notebooks so the expensive, hard-to-redo
work is separated from the cheap work you'll iterate on:

1. **`kaggle_01_results`** — the heavy run (~10 h on dual-T4): download, parse,
   audit, dose corpora, tokenizer, the 63-model training sweep, and SLOR +
   n-gram evaluation. When it finishes you have all the raw results
   (`results/eval_results.csv`, the sweep manifest, per-run perplexities) and a
   summary that tells you whether training and eval looked healthy — so you can
   decide **before** doing anything else whether you need to re-run.
2. **`kaggle_02_analysis`** — the fast run (minutes, CPU): Hill fits, the E0
   indirect-evidence index, model comparison, clustering, transfer,
   predictability, generalization, the decision rule, and all figures. It reads
   the CSVs the first notebook produced, so you can re-run and tweak the analysis
   freely without ever retraining.

They chain through Kaggle's **notebook-output datasets**: run
`kaggle_01_results` to completion, *Save Version*, then in `kaggle_02_analysis`
add that output as an input (*Add Input -> Your Datasets*). The
**restore-prior-artifacts** cell copies its `data/`, `models/`, and `results/`
into the working repo, so the analysis stages find everything they need.

Nothing here fabricates results. A blocked or failed stage produces no numbers —
it just says so in the dashboard and lets the rest proceed.

In [ ]:
# --- Setup: make the `drc` package importable, idempotently. -------------------
# Safe to re-run. If `drc` already imports we do nothing heavy. Otherwise we find
# the repo (uploaded as a dataset, or already cloned) or clone it, then install.
import os, sys, glob, subprocess
from pathlib import Path

# Replace this with your repo's clone URL (used only if the repo isn't already
# present as a Kaggle dataset or a prior checkout under /kaggle/working).
REPO_URL = "https://github.com/your-org/drc-emnlp-2026.git"  # <-- EDIT ME
WORK = Path("/kaggle/working/drc-emnlp-2026")

# Smoke runs every stage, so install the full stack (train extra covers stanza + datasets; core deps cover analysis).
EXTRA = "train"               # package extra to try, or "" for none
PIP_PACKAGES = ""  # plain pip fallbacks this notebook needs


def _have_drc() -> bool:
    try:
        import drc  # noqa: F401
        return True
    except Exception:
        return False


def _find_repo() -> Path | None:
    """Look for an existing checkout: a uploaded dataset or a prior /kaggle/working."""
    candidates = []
    # A repo uploaded as a Kaggle dataset shows up under /kaggle/input/<name>/...
    candidates += glob.glob("/kaggle/input/*/drc-emnlp-2026")
    candidates += glob.glob("/kaggle/input/*/src/drc")   # repo root contains src/drc
    candidates += [str(WORK)]
    for c in candidates:
        c = Path(c)
        # Normalise: we want the repo ROOT (the dir that contains src/drc).
        root = c
        if root.name == "drc" and root.parent.name == "src":
            root = root.parent.parent
        if (root / "src" / "drc").exists() or (root / "pyproject.toml").exists():
            return root
    return None


def _run(cmd: str) -> int:
    print("$", cmd)
    return subprocess.call(cmd, shell=True)


if _have_drc():
    print("drc already importable — skipping repo setup.")
    # Still try to locate WORK so os.chdir below lands somewhere sensible.
    found = _find_repo()
    if found is not None:
        WORK = found
else:
    repo = _find_repo()
    if repo is not None and repo != WORK:
        print(f"Found repo at {repo} (not cloning).")
        WORK = repo
    elif repo is None:
        print(f"No local repo found; cloning {REPO_URL} -> {WORK}")
        WORK.parent.mkdir(parents=True, exist_ok=True)
        _run(f'git clone --depth 1 "{REPO_URL}" "{WORK}"')
    else:
        print(f"Using existing checkout at {WORK}")

    # Try an editable install with the extra; degrade gracefully on any failure.
    installed = False
    if EXTRA:
        rc = subprocess.call(
            f'pip install -q -e "{WORK}"[{EXTRA}]', shell=True
        )
        installed = rc == 0
        if not installed:
            print(f"[warn] editable install with [{EXTRA}] failed; trying plain -e")
    if not installed:
        rc = subprocess.call(f'pip install -q -e "{WORK}"', shell=True)
        installed = rc == 0
    if not installed:
        # Last resort: don't install, just put src/ on the path so imports work.
        src = str(WORK / "src")
        if src not in sys.path:
            sys.path.insert(0, src)
        print(f"[warn] pip install failed; added {src} to sys.path as fallback.")

    # Notebook-specific plain packages (e.g. stanza, scipy) on top of the base.
    if PIP_PACKAGES.strip():
        _run(f"pip install -q {PIP_PACKAGES}")

# Work from the repo root so the config's RELATIVE paths resolve under it.
os.chdir(WORK)
print("cwd:", os.getcwd())
print("drc importable:", _have_drc())

In [ ]:
# --- Restore prior artifacts so already-done stages resume as "skipped". -------
# When this notebook is chained after another, the previous notebook's
# /kaggle/working is attached as an input dataset under /kaggle/input/<name>/.
# We copy its data/ models/ results/ into our working repo. Safe to re-run;
# dirs_exist_ok lets it merge over an existing tree. Guarded so a missing input
# (e.g. when you run this notebook standalone) is a no-op, not an error.
import glob, shutil
from pathlib import Path

WORK = Path(os.getcwd())  # set by the setup cell
RESTORE_DIRS = ("data", "models", "results")

# Candidate source roots: a chained notebook-output dataset will contain the
# repo's working tree, either at the repo root or one level down.
sources = []
sources += glob.glob("/kaggle/input/*/drc-emnlp-2026")
sources += glob.glob("/kaggle/input/*")

restored = []
for src_root in sources:
    src_root = Path(src_root)
    if src_root.resolve() == WORK.resolve():
        continue
    for sub in RESTORE_DIRS:
        src = src_root / sub
        if src.is_dir():
            try:
                shutil.copytree(src, WORK / sub, dirs_exist_ok=True)
                restored.append(str(src))
            except Exception as exc:  # never let a restore failure stop the run
                print(f"[warn] could not restore {src}: {exc}")

if restored:
    print("Restored prior artifacts from:")
    for r in restored:
        print("  ", r)
else:
    print("No prior artifacts found to restore (fine if this is the first stage).")

In [ ]:
# --- Detect GPUs and pick the sweep mode. --------------------------------------
import subprocess

n_gpus = 0
try:
    out = subprocess.run(
        ["nvidia-smi", "-L"], capture_output=True, text=True, check=False
    )
    print(out.stdout.strip() or "(nvidia-smi returned no GPUs)")
    n_gpus = sum(1 for ln in out.stdout.splitlines() if ln.strip().startswith("GPU "))
except FileNotFoundError:
    print("nvidia-smi not found — assuming no GPU (CPU-only).")

SINGLE_GPU = n_gpus < 2
print(f"\nDetected {n_gpus} GPU(s). SINGLE_GPU = {SINGLE_GPU}")

if n_gpus >= 2:
    print("Dual-GPU: the sweep runs two training jobs in parallel, one per card.")
elif n_gpus == 1:
    print("Single-GPU: the sweep uses the 48-run fallback (drops dose=4).")
else:
    print("No GPU: training/eval stages will fail; set Accelerator to GPU T4 x2.")

print(
    "\n[CAVEAT] T4 is a Turing card and does NOT support bf16. The shipped"
    "\n         configs/base.yaml uses precision: bf16. Before training on T4,"
    "\n         edit that line to  precision: fp16  (this cell does not edit it)."
)

In [ ]:
# --- Run the pipeline. ---------------------------------------------------------
# default_phases() returns the wired stages; run_pipeline() isolates failures,
# skips finished stages, blocks stages with unmet deps, and prints a dashboard.
# It never raises on a stage failure, so this cell completes even if a stage dies.
from pathlib import Path
from drc.pipeline import default_phases, run_pipeline
from drc.data.download import load_config, resolve_path

CONFIG_PATH = "configs/smoke.yaml"          # reused by the cells below
cfg = Path(CONFIG_PATH)
results_dir = resolve_path(cfg, load_config(cfg)["paths"]["results"])
status = results_dir / "pipeline_status.json"

# Run EVERY stage on the tiny smoke config.
only = None  # None == every stage

stages = default_phases(cfg, single_gpu=SINGLE_GPU)
results = run_pipeline(stages, status_path=status, only=only)
# The dashboard is already printed above by run_pipeline.

In [ ]:
# --- Inspect what we produced. -------------------------------------------------
# Tolerant of missing files: a fresh or partial run just shows fewer artifacts.
# Reads the results dir from the same config the run cell used (CONFIG_PATH).
import json
from pathlib import Path
from drc.data.download import load_config, resolve_path

results_dir = resolve_path(Path(CONFIG_PATH), load_config(CONFIG_PATH)["paths"]["results"])

status_path = results_dir / "pipeline_status.json"
if status_path.exists():
    data = json.loads(status_path.read_text())
    print("Pipeline status:")
    for name, r in data.items():
        secs = f"{r.get('seconds', 0):.1f}s" if r.get("seconds") else ""
        detail = f"  {r['detail']}" if r.get("detail") else ""
        print(f"  {r['status']:<8} {name:<18} {secs}{detail}")
else:
    print(f"No {status_path} yet — has the run cell completed?")

for d in (results_dir, results_dir / "figures"):
    if d.is_dir():
        items = sorted(x.name for x in d.iterdir())
        print(f"\n{d}/ ({len(items)} items):")
        for it in items:
            print("  ", it)
    else:
        print(f"\n{d}/ does not exist yet.")

decision = results_dir / "decision.txt"
if decision.exists():
    print("\n=== decision.txt ===")
    print(decision.read_text())

In [ ]:
# --- Results health check: should I re-run anything? ---------------------------
# Tolerant of partial runs. Surfaces failed training runs, perplexity outliers,
# and the E0 / E_max each construction landed at, so you can judge the run.
import json
from pathlib import Path
from drc.data.download import load_config, resolve_path

_paths = load_config(CONFIG_PATH)["paths"]
results_dir = resolve_path(Path(CONFIG_PATH), _paths["results"])
models_dir = resolve_path(Path(CONFIG_PATH), _paths["models"])

problems = []

# 1) Training sweep: how many runs finished, and did any fail?
manifest = results_dir / "sweep_manifest.json"
if manifest.exists():
    m = json.loads(manifest.read_text())
    runs = m.get("runs", {})
    by_status = {}
    for r in runs.values():
        by_status[r.get("status", "?")] = by_status.get(r.get("status", "?"), 0) + 1
    print("Training runs:", dict(by_status), f"(of {m.get('total_runs', len(runs))})")
    failed = [k for k, r in runs.items() if r.get("status") == "failed"]
    if failed:
        problems.append(f"{len(failed)} training run(s) failed: {failed[:5]}...")
        print("  FAILED:", failed)
else:
    print("No sweep_manifest.json — training may not have run.")

# 2) Held-out perplexities (sanity band [15, 40]).
ppls = []
for mj in sorted(models_dir.glob("ltgbert_*/metrics.json")):
    try:
        d = json.loads(mj.read_text())
        ppl = d.get("perplexity") or d.get("final_perplexity")
        if ppl is not None:
            ppls.append((mj.parent.name, float(ppl)))
    except Exception:
        pass
if ppls:
    bad = [(n, p) for n, p in ppls if not (15 <= p <= 40)]
    lo = min(p for _, p in ppls); hi = max(p for _, p in ppls)
    print(f"\nPerplexity: {len(ppls)} models, range {lo:.1f}-{hi:.1f}.")
    if bad:
        problems.append(f"{len(bad)} model(s) out of the [15,40] perplexity band.")
        print("  OUT OF BAND:", bad[:5])

# 3) E0 (dose 0) and E_max (shared 'full' model) per construction.
eval_csv = results_dir / "eval_results.csv"
if eval_csv.exists():
    import pandas as pd
    df = pd.read_csv(eval_csv)
    self_eval = df[df["model_construction"] == df["eval_construction"]]
    e0 = (self_eval[self_eval["dose"].astype(str) == "0"]
          .groupby("eval_construction")["accuracy"].mean())
    emax = (df[df["model_construction"] == "full"]
            .groupby("eval_construction")["accuracy"].mean())
    print("\nPer-construction E0 (zero exposure) and E_max (full corpus):")
    for c in sorted(set(e0.index) | set(emax.index)):
        print(f"  {c:<26} E0={e0.get(c, float('nan')):.3f}   "
              f"E_max={emax.get(c, float('nan')):.3f}")
    # Replication sanity: the full model on AANN should land ~[0.55, 0.75].
    aann_max = emax.get("aann")
    if aann_max is not None and not (0.55 <= aann_max <= 0.75):
        problems.append(f"AANN ceiling {aann_max:.2f} outside the ~[0.55,0.75] "
                        "replication band — check the eval pipeline.")
else:
    print("\nNo eval_results.csv yet — eval may not have run.")

print("\n" + "=" * 60)
if problems:
    print("RE-RUN GUIDANCE: issues found, review before trusting results:")
    for p in problems:
        print("  -", p)
else:
    print("RE-RUN GUIDANCE: no red flags. Proceed to kaggle_02_analysis.")
print("=" * 60)